# Frequentist vs Bayesian — the paradigm axis

`BayesianMMM` has **three** independent inference choices. Two of them are about
computation; this notebook is about the third, which is about *what kind of
object you end up with*.

| Axis | Set by | Question it answers | Notebook |
|---|---|---|---|
| Fit method | `fit(method=...)` | *What* do we estimate — the exact posterior, or a fast approximation? | [`approximate_posteriors.ipynb`](approximate_posteriors.ipynb) |
| NUTS backend | `fit(nuts_sampler=...)` | *How* is the NUTS trajectory computed? | [`nuts_backends.ipynb`](nuts_backends.ipynb) |
| **Paradigm** | `ModelConfig.inference_method` | Is there a **posterior at all**? | **this one** |

The first axis changes the answer. The second must not. The third changes the
*question*: a frequentist fit produces a point estimate and a sampling
distribution, and words like "there is a 90% probability the ROI is in this
range" stop being true of it.

## The question the epic opened with

> **Ridge regression is MAP estimation with Gaussian priors.** So "add ridge"
> risks duplicating a shipped capability under a new name.

That is the right challenge, and this notebook exists to answer it with numbers
rather than assertion. Three things a frequentist path can genuinely add:

1. **Transform hyperparameters by search, not by prior.** The Bayesian path
   estimates adstock α and saturation λ jointly with β. The ridge path *fixes*
   them per candidate and solves the resulting **linear** problem in closed
   form — a different estimator with different failure modes, and a fast one.
2. **Frequentist uncertainty.** A bootstrap confidence interval is not a
   posterior and cannot borrow its semantics.
3. **Hard constraints.** β ≥ 0, budget sums, monotone response. A `HalfNormal`
   prior makes a negative coefficient *unlikely*; a constraint makes it
   **impossible**.

If none of those hold up, the honest conclusion is that the scope should shrink.
We check all three below.

In [1]:
import contextlib, logging, os, time, warnings

import numpy as np
import pandas as pd

os.environ.setdefault("TQDM_DISABLE", "1")
warnings.filterwarnings("ignore")
for _n in ("pymc", "pymc.sampling", "numpyro", "jax", "arviz", "pytensor"):
    _lg = logging.getLogger(_n); _lg.setLevel(logging.ERROR); _lg.propagate = False


@contextlib.contextmanager
def quiet():
    "Hide the samplers' progress bars / chatter; our own prints stay visible."
    # Stream redirection alone is not enough: logging handlers bind their stream
    # at creation, so pymc's convergence notices bypass it. Disable logging for
    # the duration too -- we report the convergence verdict ourselves.
    prev = logging.root.manager.disable
    logging.disable(logging.CRITICAL)
    try:
        with open(os.devnull, "w") as _dn, contextlib.redirect_stdout(_dn), \
                contextlib.redirect_stderr(_dn):
            yield
    finally:
        logging.disable(prev)


from mmm_framework.config import ModelConfig
from mmm_framework.config.enums import InferenceMethod
from mmm_framework.model import BayesianMMM
from mmm_framework.model.trend_config import TrendConfig, TrendType
from mmm_framework.synth import dgp

TREND = TrendConfig(type=TrendType.LINEAR)
SEED = 11

scenario = dgp.make_clean()
panel = scenario.panel()
CHANNELS = scenario.channels

print(f"world      : {scenario.name} — {scenario.description}")
print(f"periods    : {len(scenario.weeks)}   channels: {CHANNELS}")
print(f"truth known: {scenario.representable} (the model can represent it exactly)")
scenario.true_contribution.round(1).to_frame("true incremental KPI")

world      : clean — Model's exact generative family (geometric adstock, 1-exp saturation, additive, Gaussian, constant betas, exogenous spend).
periods    : 156   channels: ['TV', 'Search', 'Social', 'Display']
truth known: True (the model can represent it exactly)


,true incremental KPI
TV,5667.6
Search,4530.9
Social,4017.2
Display,2957.9


## 1. Is ridge just MAP?

The equivalence "ridge = MAP" holds **for Gaussian coefficient priors**. It is a
statement about a specific prior, not about Bayesian estimation in general — so
the first thing to check is which priors this model actually uses.

In [2]:
mmm_probe = BayesianMMM(panel, ModelConfig(), TREND)
graph = mmm_probe.model

prior_families = {
    rv.name: rv.owner.op.__class__.__name__.replace("RV", "")
    for rv in graph.free_RVs
}
media = {k: v for k, v in prior_families.items() if k.startswith("beta_")
         and not k.startswith("beta_controls")}
print("Media coefficient priors under the default (coefficient) mode:")
for name, family in media.items():
    print(f"  {name:<18} {family}")

roi_graph = BayesianMMM(panel, ModelConfig(media_prior_mode="roi"), TREND).model
roi_families = {
    rv.name: rv.owner.op.__class__.__name__.replace("RV", "")
    for rv in roi_graph.free_RVs if rv.name.startswith("roi_")
}
print("\nUnder media_prior_mode='roi' (the agent default):")
for name, family in roi_families.items():
    print(f"  {name:<18} {family}")

Media coefficient priors under the default (coefficient) mode:
  beta_TV            Gamma
  beta_Search        Gamma
  beta_Social        Gamma
  beta_Display       Gamma

Under media_prior_mode='roi' (the agent default):
  roi_TV             LogNormal
  roi_Search         LogNormal
  roi_Social         LogNormal
  roi_Display        LogNormal


Neither is Gaussian — the coefficient prior is `Gamma(mu=1.5, sigma=1)` and the
ROI prior is `LogNormal(0, 1)`. So **MAP on this model is a different penalized
estimator than ridge**, not the same one under another name. The equivalence
would hold if you deliberately set Normal coefficient priors; it does not hold
for the shipped defaults.

That settles the *definitional* question. The interesting one is empirical: do
the two land in the same place anyway?

## 2. Three estimators, one answer key

All three are graded on the same quantity — per-channel total incremental KPI,
read from the model's own `channel_contributions` deterministic — against
`Scenario.true_contribution`. Using the model's own decomposition for every
estimator matters: otherwise the comparison measures three different definitions
of "contribution" rather than three estimators.

In [3]:
def contributions(mmm) -> np.ndarray:
    cc = np.asarray(mmm._trace.posterior["channel_contributions"].values)
    flat = cc.reshape(-1, *cc.shape[2:])          # (draws, obs, channel)
    return (flat.sum(axis=1) * mmm.y_std).mean(axis=0)


def grade(label, mmm, elapsed, truth, extra=None):
    got = contributions(mmm)
    true = np.asarray([truth[c] for c in CHANNELS], dtype=float)
    rel = np.abs(got - true) / np.maximum(np.abs(true), 1e-9)
    row = {
        "estimator": label,
        "seconds": round(elapsed, 1),
        "mean |rel err|": round(float(rel.mean()), 3),
        "worst channel": CHANNELS[int(rel.argmax())],
        "worst |rel err|": round(float(rel.max()), 3),
    }
    row.update(extra or {})
    return row


def timed(fn):
    t0 = time.perf_counter()
    with quiet():
        out = fn()
    return out, time.perf_counter() - t0

In [4]:
rows = []

# --- ridge: the frequentist path -----------------------------------------
ridge_cfg = ModelConfig(
    inference_method=InferenceMethod.FREQUENTIST_RIDGE,
    bootstrap_samples=300,
    optim_maxiter=128,
)
mmm_ridge = BayesianMMM(panel, ridge_cfg, TREND)
res_ridge, t_ridge = timed(lambda: mmm_ridge.fit(random_seed=SEED))
rows.append(grade("ridge", mmm_ridge, t_ridge, scenario.true_contribution,
                  {"converged": res_ridge.converged}))

# --- MAP: the Bayesian point estimate ------------------------------------
mmm_map = BayesianMMM(panel, ModelConfig(), TREND)
res_map, t_map = timed(lambda: mmm_map.fit(method="map", random_seed=SEED))
rows.append(grade("map", mmm_map, t_map, scenario.true_contribution,
                  {"converged": res_map.converged}))

# --- NUTS: the full posterior --------------------------------------------
mmm_nuts = BayesianMMM(
    panel, ModelConfig(n_draws=1000, n_tune=1000, n_chains=4), TREND
)
res_nuts, t_nuts = timed(lambda: mmm_nuts.fit(random_seed=SEED))
rows.append(grade("nuts", mmm_nuts, t_nuts, scenario.true_contribution,
                  {"converged": res_nuts.converged}))

pd.DataFrame(rows).set_index("estimator")

Output()

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

,seconds,mean |rel err|,worst channel,worst |rel err|,converged
estimator,,,,,
ridge,3.0,0.054,Search,0.132,None
map,2.6,0.073,Search,0.118,None
nuts,10.7,0.076,Search,0.121,True


Read the `converged` column carefully. NUTS reports `True` or `False`; the other
two report **`None`** — *not assessable*, which is not the same as "passed".

For MAP the reason is that there is no chain to assess. For ridge the reason is
stronger: R-hat and effective sample size are properties of an MCMC sampler, and
this fit ran none. That `None` is load-bearing. A `(chain=1, draw=n_boot)`
bootstrap trace passes every convergence check as `True` if you let it — R-hat
on one chain is `NaN` (filtered to `None`, which raises no flag) and effective
sample size comes back at ≈`n_boot` because bootstrap replicates are iid. The
framework branches on the estimation family precisely so that a bootstrap fit
cannot render a green convergence verdict.

## 3. What the intervals mean

The frequentist fit stamps its own provenance. This is what every report surface
reads to decide both its vocabulary and which sections it is entitled to show.

In [5]:
d = res_ridge.diagnostics
for key in ("inference_family", "estimator", "interval_kind", "interval_semantics",
            "selection_criterion", "block_length", "residual_rho", "n_boot",
            "effective_dof", "n_params", "penalty"):
    print(f"{key:<22} {d.get(key)}")

print("\nCaveats that ride with every rendered number:\n")
for c in d["caveats"]:
    print(" •", c, "\n")

inference_family       frequentist
estimator              ridge
interval_kind          bootstrap_percentile
interval_semantics     conditional_on_selection
selection_criterion    rolling_origin_mape
block_length           2
residual_rho           0.10360591822361008
n_boot                 300
effective_dof          10.790948142519762
n_params               11
penalty                0.1

Caveats that ride with every rendered number:

 • These are bootstrap CONFIDENCE intervals from a frequentist point estimate, not credible intervals: they describe the sampling variability of the estimator, not a probability distribution over the parameter. 'There is a 90% probability the ROI is in this range' is false for them. 

 • Ridge is biased by construction, so a percentile interval covers the estimator's sampling distribution rather than the true parameter. The penalty is currently using 10.8 of 11 effective degrees of freedom — close to unpenalized, so shrinkage bias is small. 

 • The adstock

Three things are worth pausing on.

**`interval_kind = bootstrap_percentile`.** These are **confidence** intervals.
They describe how much the estimate would move across resamples of the data —
not a probability distribution over the parameter. The sentence "there is a 90%
probability the ROI is in this range" is true of a credible interval and false
of this one, and the reporting layer says so on every surface.

**`block_length`.** MMM residuals are serially correlated, and an iid residual
bootstrap treats each week as exchangeable — which understates the variance and
produces intervals that are too narrow. The block length is *estimated* from the
residual autocorrelation rather than assumed, so an uncorrelated dataset pays no
width penalty (`block_length` collapses to 1) while a correlated one is widened.
Measured over 60 simulations of this world with AR(1) errors at ρ = 0.6, 90%
intervals covered **79.6%** under the iid bootstrap and **90.4%** under blocks.

**`interval_semantics = conditional_on_selection`.** The transforms and the
penalty were chosen once by search, and every replicate conditions on that
choice — so this interval **omits selection uncertainty and is too narrow**.
That is a correctness tradeoff, not a precision one: a cheap interval that is
merely *wider* than necessary is conservative and self-announcing, while this one
is narrower, and narrowness reads as confidence. `refit_search=True` re-runs the
search inside every replicate and is what a published interval should use.

In [6]:
# What the conditioning is hiding, on this dataset: re-run the search inside
# every replicate and compare interval width. Deliberately small so it finishes;
# a real run uses the full budget.
from mmm_framework.frequentist import bootstrap_fit

SMALL = {"budget": 24, "horizon": 13, "max_origins": 2}

def window_width(idata, level=0.9):
    cc = np.asarray(idata.posterior["channel_contributions"].values[0])
    totals = cc.sum(axis=1)                       # (draw, channel)
    lo, hi = np.percentile(totals, [(1 - level) / 2 * 100,
                                    (1 + level) / 2 * 100], axis=0)
    return pd.Series(hi - lo, index=CHANNELS)

widths = {}
for label, refit in (("conditional (default)", False), ("selection resampled", True)):
    with quiet():
        idata, diag = bootstrap_fit(
            panel, model_config=ridge_cfg, trend_config=TREND,
            n_boot=60, refit_search=refit, search_kwargs=SMALL, seed=SEED,
        )
    widths[label] = window_width(idata)
    print(f"{label:<24} semantics={diag['interval_semantics']}")

w = pd.DataFrame(widths)
w["inflation ×"] = (w["selection resampled"] / w["conditional (default)"]).round(2)
w.round(1)

conditional (default)    semantics=conditional_on_selection


selection resampled      semantics=selection_resampled


,conditional (default),selection resampled,inflation ×
TV,17.2,75.7,4.4
Search,19.5,147.7,7.6
Social,19.9,68.2,3.4
Display,20.6,44.0,2.1


The inflation factor is the price of the cheap default, measured on *this*
dataset. It is a property of the data — how much the criterion can order the
candidate transforms — not a constant the library could publish in advance,
which is exactly why the label ships rather than a correction factor.

## 4. What the search recovers, and what it cannot

`make_clean` plants per-channel carryover and saturation, so the search can be
graded directly rather than argued about.

In [7]:
from mmm_framework.frequentist.search import search_transforms
from mmm_framework.synth.dgp import _ALPHA, _LAM

result, t_search = timed(
    lambda: search_transforms(panel, model_config=ModelConfig(),
                              trend_config=TREND, budget=256, seed=0)
)
print(f"search: {len(result.candidates)} candidates in {t_search:.1f}s "
      f"({result.criterion})")

near = result.spread(0.10)          # candidates the data cannot order
recovery = []
for ch in CHANNELS:
    a_found = result.best.alpha[ch]["alpha"]
    l_found = result.best.lam[ch]["sat_lam"]
    l_near = [c.lam[ch]["sat_lam"] for c in near]
    recovery.append({
        "channel": ch,
        "α true": _ALPHA[ch], "α found": round(a_found, 2),
        "|α err|": round(abs(a_found - _ALPHA[ch]), 2),
        "λ true": _LAM[ch], "λ found": round(l_found, 2),
        "λ near-optimal range": f"{min(l_near):.2f} – {max(l_near):.2f}",
    })
print(f"{len(near)} of {len(result.candidates)} candidates score within 10% of the best\n")
pd.DataFrame(recovery).set_index("channel")

search: 256 candidates in 2.9s (rolling_origin_mape)
28 of 256 candidates score within 10% of the best



,α true,α found,|α err|,λ true,λ found,λ near-optimal range
channel,,,,,,
TV,0.6,0.67,0.07,1.6,1.77,0.72 – 2.28
Search,0.2,0.22,0.02,1.8,1.39,0.76 – 2.30
Social,0.4,0.14,0.26,1.7,0.91,0.70 – 2.29
Display,0.5,0.71,0.21,1.5,1.77,0.74 – 2.14


**Carryover is recovered; saturation is not.** The λ column tells the story: the
candidates the criterion cannot distinguish span most of the allowed range, so
the winner's λ is close to a draw from that set rather than an estimate.

This is not a defect of the search, and it is not particular to the frequentist
path. Jin et al. (Google, 2017) found the Hill parameters "essentially
unidentifiable in some scenarios"; Dew et al. (2024) show predictive fit —
cross-validation included — cannot arbitrate between observationally equivalent
response specifications. No production MMM identifies saturation from the sales
likelihood; each one accommodates the fact. The Bayesian path accommodates it
with a prior, this path accommodates it by **bounding in data units** (the
half-saturation point is confined to a fraction of observed maximum spend, the
same move Robyn and Meridian make).

What genuinely identifies curvature is **dose spread** — a rank condition on the
spend design, which is a property of the data, not of the estimator. Read
`SearchResult.spread()`, not the winner.

## 5. Constraints — the capability a prior cannot express

This is the clearest thing the frequentist path adds. A `HalfNormal` prior makes
a negative media coefficient *unlikely*; a constraint makes it **impossible**.
And a constraint on the *sum* of contributions — "media accounted for exactly
the $X finance already booked" — has no prior analogue at all.

In [8]:
try:
    import cvxpy  # noqa: F401
    HAVE_CVXPY = True
except ImportError:
    HAVE_CVXPY = False
    print("cvxpy not installed — pip install 'mmm-framework[frequentist]'")

if HAVE_CVXPY:
    from mmm_framework.frequentist import build_design_matrix, fit_constrained, fit_ridge
    from mmm_framework.frequentist.constrained import nonneg, sum_at_most

    alpha = {c: dict(result.best.alpha[c]) for c in CHANNELS}
    lam = {c: dict(result.best.lam[c]) for c in CHANNELS}
    design = build_design_matrix(panel, alpha, lam,
                                 model_config=ModelConfig(), trend_config=TREND)
    media_cols = design.columns[design.blocks["media"]]

    free = fit_ridge(design, penalty=result.best.penalty)
    signed = fit_constrained(design, penalty=result.best.penalty,
                             constraints=nonneg(design))
    capped = fit_constrained(
        design, penalty=result.best.penalty,
        constraints=nonneg(design) + [sum_at_most(design, list(media_cols),
                                                  0.8 * float(design.y.sum()))],
    )

    comp = pd.DataFrame({
        "unconstrained": free.as_dict(),
        "β ≥ 0": signed.as_dict(),
        "β ≥ 0 + capped total": capped.as_dict(),
    }).loc[list(media_cols)].round(3)
    print("Active constraints under the cap:", capped.active)
    print("Coefficients pinned at a boundary (no meaningful two-sided interval):",
          [c for c, on in zip(design.columns, capped.at_boundary) if on])
    comp

Active constraints under the cap: ['media_TV >= 0', 'media_Search >= 0', 'media_Social >= 0', 'media_Display >= 0']
Coefficients pinned at a boundary (no meaningful two-sided interval): ['media_TV', 'media_Search', 'media_Social', 'media_Display']


A coefficient pinned by an active constraint has **no meaningful two-sided
interval** — the bootstrap reports which ones those are in
`diagnostics["at_boundary"]`, because a replicate distribution that piles up on
a boundary is not describing sampling variability of an interior estimate.

`ModelConfigBuilder().frequentist_cvxpy()` selects this estimator end-to-end;
with no explicit constraints it applies non-negative media, the one restriction
every MMM wants.

## 6. Does it fail *differently*?

The most useful question in the epic. A fast estimator that fails the same way
as the Bayesian path is redundant — you would just run the Bayesian one. One
that fails *differently* is a triangulation tool: disagreement between them is
information about the specification rather than noise.

Three worlds where the model's assumptions are broken on purpose.

In [9]:
VIOLATIONS = [
    ("unobserved_confounding", dgp.make_unobserved_confounding,
     "latent demand drives BOTH spend and KPI"),
    ("adstock_misspec", dgp.make_adstock_misspec, "carryover shape is wrong"),
    ("saturation_misspec", dgp.make_saturation_misspec, "response curve is wrong"),
]

def signed_bias(mmm, truth, channels):
    got = contributions(mmm)
    true = np.asarray([truth[c] for c in channels], dtype=float)
    return float(np.mean((got - true) / np.maximum(np.abs(true), 1e-9)) * 100)

viol = []
for name, factory, note in VIOLATIONS:
    sc = factory()
    p = sc.panel()
    chans = sc.channels
    row = {"world": name, "breaks": note}
    for label, cfg, kw in (
        ("ridge", ModelConfig(inference_method=InferenceMethod.FREQUENTIST_RIDGE,
                              bootstrap_samples=200, optim_maxiter=96), {}),
        ("map", ModelConfig(), {"method": "map"}),
        ("nuts", ModelConfig(n_draws=500, n_tune=500, n_chains=2), {}),
    ):
        try:
            m = BayesianMMM(p, cfg, TREND)
            with quiet():
                m.fit(random_seed=SEED, **kw)
            got = np.asarray([
                np.asarray(m._trace.posterior["channel_contributions"].values)
                .reshape(-1, m.n_obs, len(chans))[:, :, j].sum(axis=1).mean()
                * m.y_std for j in range(len(chans))
            ])
            true = np.asarray([sc.true_contribution[c] for c in chans])
            rel = np.abs(got - true) / np.maximum(np.abs(true), 1e-9)
            row[f"{label} |err|"] = round(float(rel.mean()), 3)
            row[f"{label} bias %"] = round(
                float(np.mean((got - true) / np.maximum(np.abs(true), 1e-9)) * 100), 1
            )
        except Exception as exc:                # a failure is a reported row
            row[f"{label} |err|"] = f"failed: {type(exc).__name__}"
            row[f"{label} bias %"] = None
    viol.append(row)
    print(f"  {name} done", flush=True)

pd.DataFrame(viol).set_index("world")

Output()

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  unobserved_confounding done


Output()

...

Output()

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  saturation_misspec done


,breaks,ridge |err|,ridge bias %,map |err|,map bias %,nuts |err|,nuts bias %
world,,,,,,,
unobserved_confounding,latent demand drives BOTH spend and KPI,0.495,26.5,0.297,5.9,0.305,7.7
adstock_misspec,carryover shape is wrong,0.930,-76.5,0.954,-95.4,0.812,-81.2
saturation_misspec,response curve is wrong,0.688,68.8,0.399,39.9,0.388,38.8


Read the **bias** columns, not just the errors. Under unobserved confounding all
three should over-credit media in the same direction, because the bias is
*causal* — it lives in the data, not in the estimator, and no amount of
regularization or sampling fixes a back-door path. That agreement is itself the
finding: **if the three disagree, the disagreement is about specification; if
they agree and are wrong, the problem is identification**, and the fix is an
experiment (`mmm_framework.calibration`), not a different estimator.

Under transform misspecification the picture is more interesting, because that
is precisely where the two paradigms differ in *mechanism*: the Bayesian path
estimates the transform under a prior, the frequentist path selects it by
out-of-sample error. Whether that produces a materially different answer is a
property of the dataset, which is why it is measured here rather than claimed.

## 7. The verdict

Written down plainly, because the epic asked for it and because "it depends"
would be a non-answer.

**Choose the frequentist path when:**

- **you need a hard constraint.** β ≥ 0, an ordering between channels, or a
  total that must match a booked number. This is the one capability a prior
  genuinely cannot express, and it is the strongest standalone justification in
  the epic.
- **you want a second, mechanically different read** on the same data. It
  selects transforms by predictive error rather than estimating them under a
  prior, so agreement is evidence and disagreement is a specification question
  worth chasing. Triangulation, not replacement.
- **you are iterating on specification** and a full posterior per iteration is
  too slow. Measure this on your own data — the table in section 2 is one world
  at one size.

**Stay on the Bayesian path when:**

- **you will publish the intervals.** A conditional-on-selection bootstrap
  interval is too narrow, `refit_search=True` is expensive, and ridge shrinkage
  bias means a percentile interval covers the *estimator's* sampling
  distribution rather than the true parameter — measured in #186, where one
  channel sat below nominal even with the block correction working.
- **you want to encode knowledge.** Experiment calibration, ROI priors,
  hierarchical pooling across geographies, partial pooling of a channel group —
  all of these are prior-shaped and have no frequentist counterpart here.
- **the model is not linear given fixed transforms.** A GP trend, per-geo media
  coefficients, time-varying coefficients or reach/frequency channels are
  refused outright, naming the feature — rather than silently dropping the term
  and returning a number that looks like the model's answer.

**What it is not:** a faster way to get the same answer. The point estimate is a
different estimator, the interval is a different object, and a report generated
from one says different words than a report generated from the other. That is
the whole reason the paradigm is a separate axis.